# 🎬 OpenSource Clipping — Optimized Colab Notebook

**3-Cell Architecture:** ① Setup → ② Config → ③ Runtime

**New Features:**
- **Ollama** support (local/remote LLM inference)
- **YouTube Transcript API** (skip Whisper entirely when available)
- **Per-clip overrides** (different ratio/font/hook/broll per rank)
- **Structured logging** with emoji levels
- **VRAM monitoring** + aggressive cache clearing between stages
- **Drive model caching** (persist across sessions)

## ① SETUP — Mount Drive, Clone Repo, Install, Verify

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════════
import os, subprocess, sys, shutil
from pathlib import Path

# ── 1.1 Mount Google Drive (for persistent model cache) ──
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── 1.2 System diagnostics ──
def print_sys_info():
    print("═" * 60)
    try:
        gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.used,memory.total","--format=csv,noheader,nounits"],
                             capture_output=True, text=True, check=False)
        name, used, total = gpu.stdout.strip().split(", ")
        print(f"🎮 GPU : {name} | VRAM {float(used):.0f}/{float(total):.0f} MB")
    except Exception as e:
        print(f"⚠️ GPU read error: {e}")
    try:
        mem = dict((i.split()[0].rstrip(":"), int(i.split()[1])) for i in open("/proc/meminfo").read().splitlines())
        print(f"🧠 RAM : {mem.get('MemAvailable',mem.get('MemFree',0))/1024**2:.1f} GB free / {mem.get('MemTotal',0)/1024**2:.1f} GB total")
    except Exception as e:
        print(f"⚠️ RAM read error: {e}")
    stat = shutil.disk_usage("/content")
    print(f"💾 Disk: {stat.free/1024**3:.1f} GB free / {stat.total/1024**3:.1f} GB total")
    print("═" * 60)

print_sys_info()

# ── 1.3 Clone / update repo ──
REPO_URL = "https://github.com/NaufalRizqullah/opensource-clipping.git"
CLONE_DIR = "/content/opensource-clipping"

if os.path.isdir(f"{CLONE_DIR}/.git"):
    print("📁 Repo exists — pulling latest…")
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only"], check=True)
else:
    print("⬇️  Cloning repo…")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CLONE_DIR], check=True)

%cd {CLONE_DIR}

# ── 1.4 Install dependencies ──
print("⏳ Installing dependencies (this takes ~2-3 min)…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("✅ Dependencies installed.")

# ── 1.5 Verify torch CUDA ──
import torch
print(f"🔥 PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device: {torch.cuda.get_device_name(0)}")

# ── 1.6 Restore cached models from Drive ──
DRIVE_CACHE = Path("/content/drive/MyDrive/osc_cache")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
MODELS = ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]
for m in MODELS:
    src, dst = DRIVE_CACHE / m, Path(CLONE_DIR) / m
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        print(f"📥 Restored {m} from Drive cache")

# ── 1.7 Load API keys from Colab Secrets (with direct-input fallback) ──
from google.colab import userdata

def get_secret(key, optional=False, allow_input=True):
    """Load from Colab Secrets; fallback to direct input if missing."""
    try:
        return userdata.get(key)
    except Exception:
        if optional:
            print(f"   ℹ️ Optional secret {key} not set.")
            return ""
        if allow_input:
            print(f"⚠️ Secret {key} tidak ditemukan di Colab Secrets.")
            print("   Cara menambahkan: klik ikon 🔑 di sidebar kiri → Add secret")
            val = input(f"   Atau paste {key} langsung di sini (lalu Enter): ").strip()
            if val:
                return val
        raise RuntimeError(
            f"Secret {key} wajib diisi.\n"
            "   1. Klik ikon 🔑 di sidebar kiri Colab\n"
            f"   2. Klik Add secret\n"
            f"   3. Name: {key}\n"
            "   4. Value: (paste API key kamu)\n"
            "   5. Centang Notebook access\n"
            "   6. Run cell ini lagi"
        )

GOOGLE_API_KEY = get_secret("GOOGLE_API_KEY", optional=False)
HF_TOKEN       = get_secret("HF_TOKEN",       optional=True)
PEXELS_API_KEY = get_secret("PEXELS_API_KEY", optional=True)
NVIDIA_API_KEY = get_secret("NVIDIA_API_KEY", optional=True)
OLLAMA_API_KEY = get_secret("OLLAMA_API_KEY", optional=True)

env_text = f"""GOOGLE_API_KEY={GOOGLE_API_KEY}
HF_TOKEN={HF_TOKEN}
PEXELS_API_KEY={PEXELS_API_KEY}
NVIDIA_API_KEY={NVIDIA_API_KEY}
OLLAMA_API_KEY={OLLAMA_API_KEY}
"""
Path(".env").write_text(env_text, encoding="utf-8")
print("🔐 .env written successfully.")
print_sys_info()

## ② CONFIG — ALL Parameters in One Place

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — CONFIGURATION (edit everything below)
# ═══════════════════════════════════════════════════════════════════
import json
from pathlib import Path

# ── Source ──
URL_VIDEO        = "https://www.youtube.com/watch?v=Dc4_aBFAYWE"
SOURCE_PLATFORM  = "youtube"      # youtube | tiktok | instagram | gdrive
SOURCE_HEIGHT    = "max"          # max | 1080 | 1440 | 2160

# ── Output ──
JUMLAH_CLIP      = 5                # how many highlight clips to generate
RASIO            = "9:16"         # 9:16 | 16:9 | 1:1 | 3:4 | 4:5
RENDER_HEIGHT    = "1080"         # 1080 | 1440 | 2160 | source

# ── AI Provider ──
AI_PROVIDER      = "gemini"       # gemini | nvidia | ollama
GEMINI_MODEL     = "gemini-3-flash-preview"
OLLAMA_URL       = "http://localhost:11434"  # change if using remote Ollama
OLLAMA_MODEL     = "llama3.1"                   # e.g. mistral, phi4, codellama
# When Gemini fails (rate-limit, unavailable model, etc.), auto-fallback to Ollama:
OLLAMA_FALLBACK_URL   = "http://localhost:11434"  # can be different from primary Ollama
OLLAMA_FALLBACK_MODEL = "gemini-3-flash-preview:cloud"  # default fallback model

# ── Transcription Speed-Up ──
USE_YT_TRANSCRIPT_API = False     # use youtube-transcript-api (fastest, skips Whisper)
USE_DLP_SUBS        = True        # use yt-dlp auto-captions when available
WHISPER_MODEL       = "large-v3"
WHISPER_COMPUTE     = "float16" # float16 (T4 GPU) | float32 (CPU fallback)

# ── Visual Style ──
FONT_STYLE       = "DEFAULT"    # DEFAULT | HORMOZI | STORYTELLER | CINEMATIC
FACE_DETECTOR    = "mediapipe"   # mediapipe (light VRAM) | yolo (sharp)
YOLO_SIZE        = "8m"          # 8n | 8s | 8m (only if face_detector=yolo)

# ── Content Toggles (True = faster on Colab) ──
NO_BGM           = True
NO_BROLL         = True
NO_SUBS          = False
NO_KARAOKE       = False
NO_HOOK          = False

# ── Podcast Modes ──
SPLIT_SCREEN     = False
SPLIT_TRIGGER    = "diarization" # diarization | face
CAMERA_SWITCH    = False

# ── Hook & Segments ──
HOOK_V2          = False
NO_SEGMENT_TRIM  = False
SILENCE_TRIM     = False

# ── Video Quality ──
VIDEO_PRESET     = "ultrafast"   # ultrafast | fast | slow | veryslow
VIDEO_SCALE_ALGO = "bicubic"     # bicubic | lanczos | bilinear
VIDEO_CQ         = 23
VIDEO_CRF        = 20
VIDEO_SHARPEN    = False

# ── Colab Optimizations ──
COLAB_CLEANUP    = True           # free GPU memory between heavy stages

# ── 🎛️ Per-Clip Overrides (optional) ──
# Set to None to disable, or provide a dict keyed by rank (1, 2, 3...)
CLIP_OVERRIDES = None
# Example:
# CLIP_OVERRIDES = {
#     "1": {"ratio": "9:16",  "font_style": "HORMOZI",    "use_broll": True,  "video_cq": 23},
#     "2": {"ratio": "1:1",    "font_style": "CINEMATIC",  "no_subs": True,    "static_crop": True},
#     "3": {"ratio": "9:16",  "use_split_screen": True,    "split_trigger": "face", "font_style": "STORYTELLER"},
#     "4": {"ratio": "16:9",  "font_style": "DEFAULT",   "video_preset": "slow", "video_scale_algo": "lanczos"},
# }

CLIP_CONFIG_PATH = None
if CLIP_OVERRIDES:
    CLIP_CONFIG_PATH = str(Path(CLONE_DIR) / "clip_override.json")
    Path(CLIP_CONFIG_PATH).write_text(json.dumps(CLIP_OVERRIDES, indent=2), encoding="utf-8")
    print(f"🎛️  Per-clip overrides written → {CLIP_CONFIG_PATH}")
else:
    print("ℹ️ Using global defaults for all clips.")

print("✅ Configuration ready.")

## ③ RUNTIME — Execute Pipeline with Live Logging

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — RUNTIME
# ═══════════════════════════════════════════════════════════════════
import subprocess, sys, time
from pathlib import Path

cmd = [
    sys.executable, "main.py",
    "--url", URL_VIDEO,
    "--source", SOURCE_PLATFORM,
    "--clips", str(JUMLAH_CLIP),
    "--ratio", RASIO,
    "--render-height", str(RENDER_HEIGHT),
    "--source-height", str(SOURCE_HEIGHT),
    "--font-style", FONT_STYLE,
    "--face-detector", FACE_DETECTOR,
    "--yolo-size", YOLO_SIZE,
    "--ai-provider", AI_PROVIDER,
    "--gemini-model", GEMINI_MODEL,
    "--ollama-url", OLLAMA_URL,
    "--ollama-model", OLLAMA_MODEL,
    "--ollama-fallback-url", OLLAMA_FALLBACK_URL,
    "--ollama-fallback-model", OLLAMA_FALLBACK_MODEL,
    "--whisper-model", WHISPER_MODEL,
    "--whisper-compute-type", WHISPER_COMPUTE,
    "--video-preset", VIDEO_PRESET,
    "--video-scale-algo", VIDEO_SCALE_ALGO,
    "--video-cq", str(VIDEO_CQ),
    "--video-crf", str(VIDEO_CRF),
    "--words-per-sub", str(5),
    "--hook-duration", str(3),
    "--colab-cleanup",
]

if USE_YT_TRANSCRIPT_API:
    cmd.append("--use-yt-transcript-api")
if USE_DLP_SUBS:
    cmd.append("--use-dlp-subs")
if NO_BGM:
    cmd.append("--no-bgm")
if NO_BROLL:
    cmd.append("--no-broll")
if NO_SUBS:
    cmd.append("--no-subs")
if NO_KARAOKE:
    cmd.append("--no-karaoke")
if NO_HOOK:
    cmd.append("--no-hook")
if SPLIT_SCREEN:
    cmd.append("--split-screen")
    cmd += ["--split-trigger", SPLIT_TRIGGER]
if CAMERA_SWITCH:
    cmd.append("--camera-switch")
if HOOK_V2:
    cmd.append("--hook-v2")
if NO_SEGMENT_TRIM:
    cmd.append("--no-segment-trim")
if SILENCE_TRIM:
    cmd.append("--silence-trim")
if VIDEO_SHARPEN:
    cmd.append("--video-sharpen")
if CLIP_CONFIG_PATH:
    cmd += ["--clip-config", CLIP_CONFIG_PATH]

print("🚀 Starting pipeline…")
print("Command:", " ".join(cmd))
print("─" * 60)

start = time.time()
result = subprocess.run(cmd, cwd=CLONE_DIR)
elapsed = time.time() - start

print("─" * 60)
print(f"⏱️ Total time: {elapsed/60:.1f} minutes")

out_dir = Path(CLONE_DIR) / "outputs"
if out_dir.exists():
    files = sorted(out_dir.glob("*_ready.mp4")) + sorted(out_dir.glob("*.jpg"))
    print(f"\n📦 Output files ({len(files)}):")
    for f in files:
        size_mb = f.stat().st_size / (1024*1024)
        print(f"   • {f.name} ({size_mb:.1f} MB)")
else:
    print("⚠️ outputs/ directory not found.")

# ── Zip outputs ──
if out_dir.exists() and list(out_dir.iterdir()):
    zip_path = Path(CLONE_DIR) / "outputs.zip"
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root_dir=str(out_dir))
    print(f"\n🤐 outputs.zip → {zip_path.stat().st_size/1024/1024:.1f} MB")
    from google.colab import files
    files.download(str(zip_path))

# ── Persist models to Drive cache ──
copied = 0
for m in ["blaze_face_full_range.tflite", "face_yolov8m.pt", "face_yolov8n.pt", "face_yolov8s.pt"]:
    src = Path(CLONE_DIR) / m
    dst = DRIVE_CACHE / m
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)
        copied += 1
        print(f"💾 Cached {m} → Drive")
if copied:
    print(f"✅ {copied} model(s) cached to Drive for next session.")

## 🆘 Emergency Cleanup (run if disk almost full)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# EMERGENCY CLEANUP — free temp files instantly
# ═══════════════════════════════════════════════════════════════════
import shutil
from pathlib import Path

patterns = ["*.ts", "*_silent_*.mp4", "*.ass", "temp_broll_*", "video_asli.mp4", "*.wav", "bgm_*.mp3"]
removed = sum(1 for pat in patterns for fp in Path(CLONE_DIR).glob(pat) if fp.unlink() or True)
print(f"🗑️ Removed {removed} temp files.")
stat = shutil.disk_usage("/content")
print(f"💾 Disk: {stat.free/1024**3:.1f} GB free / {stat.total/1024**3:.1f} GB total")